In [3]:
# Ячейка 1: Установка библиотек
!pip install ipywidgets pandas plotly

# Ячейка 2: Импорт и код CRM
import pandas as pd
import sqlite3
import plotly.express as px
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Создаём базу данных
conn = sqlite3.connect('crm.db')
c = conn.cursor()

# Создаём таблицы
c.execute('''CREATE TABLE IF NOT EXISTS clients
             (id INTEGER PRIMARY KEY AUTOINCREMENT,
              name TEXT, phone TEXT, email TEXT,
              registration_date TEXT, segment TEXT)''')

c.execute('''CREATE TABLE IF NOT EXISTS sales
             (id INTEGER PRIMARY KEY AUTOINCREMENT,
              client_id INTEGER, sale_date TEXT,
              amount REAL, product_category TEXT)''')

# Добавляем тестовые данные (если таблицы пустые)
c.execute("SELECT COUNT(*) FROM clients")
if c.fetchone()[0] == 0:
    test_clients = [
        ('Иванов Иван', '+7-999-123-45-67', 'ivan@mail.ru', '2025-01-15', 'VIP'),
        ('Петрова Мария', '+7-999-234-56-78', 'maria@mail.ru', '2025-01-20', 'Обычный'),
        ('Сидоров Алексей', '+7-999-345-67-89', 'alex@mail.ru', '2025-02-01', 'VIP'),
        ('Козлова Елена', '+7-999-456-78-90', 'elena@mail.ru', '2025-02-10', 'Обычный'),
    ]
    for client in test_clients:
        c.execute("INSERT INTO clients (name, phone, email, registration_date, segment) VALUES (?,?,?,?,?)", client)

    for i in range(1, 5):
        c.execute("INSERT INTO sales (client_id, sale_date, amount, product_category) VALUES (?,?,?,?)",
                 (i, '2025-03-15', 5000, 'Электроника'))
        c.execute("INSERT INTO sales (client_id, sale_date, amount, product_category) VALUES (?,?,?,?)",
                 (i, '2025-03-20', 3000, 'Одежда'))

conn.commit()

# Функция расчёта KPI
def show_kpi():
    clients = pd.read_sql_query("SELECT * FROM clients", conn)
    sales = pd.read_sql_query("SELECT * FROM sales", conn)

    if len(sales) > 0:
        ltv = sales.groupby('client_id')['amount'].sum().mean()
        avg_check = sales['amount'].mean()
        revenue = sales['amount'].sum()
        conversion = (sales['client_id'].nunique() / len(clients) * 100) if len(clients) > 0 else 0
    else:
        ltv = avg_check = revenue = conversion = 0

    print(f"LTV: {ltv:,.0f} RUB")
    print(f"Average check: {avg_check:,.0f} RUB")
    print(f"Conversion: {conversion:.1f}%")
    print(f"Total revenue: {revenue:,.0f} RUB")

    if len(sales) > 0:
        sales_by_cat = sales.groupby('product_category')['amount'].sum().reset_index()
        fig = px.bar(sales_by_cat, x='product_category', y='amount', title='Sales by Category')
        fig.show()

# Функция показа клиентов
def show_clients():
    clients = pd.read_sql_query("SELECT * FROM clients", conn)
    display(clients)

# Функция добавления клиента
def add_client_ui():
    name = widgets.Text(description='Full name:')
    phone = widgets.Text(description='Phone:')
    segment = widgets.Dropdown(options=['VIP', 'Regular', 'New'], description='Segment:')
    button = widgets.Button(description='Add Client')
    output = widgets.Output()

    def on_button_click(b):
        with output:
            c.execute("INSERT INTO clients (name, phone, email, registration_date, segment) VALUES (?,?,?,?,?)",
                     (name.value, phone.value, '', datetime.now().strftime('%Y-%m-%d'), segment.value))
            conn.commit()
            print(f"Client {name.value} added successfully!")

    button.on_click(on_button_click)
    display(name, phone, segment, button, output)

# Функция добавления продажи
def add_sale_ui():
    clients_df = pd.read_sql_query("SELECT id, name FROM clients", conn)
    client_options = {row['name']: row['id'] for _, row in clients_df.iterrows()}

    client = widgets.Dropdown(options=list(client_options.keys()), description='Client:')
    amount = widgets.IntText(description='Amount (RUB):', value=1000)
    category = widgets.Dropdown(options=['Electronics', 'Clothing', 'Food', 'Cosmetics', 'Home'], description='Category:')
    button = widgets.Button(description='Add Sale')
    output = widgets.Output()

    def on_button_click(b):
        with output:
            c.execute("INSERT INTO sales (client_id, sale_date, amount, product_category) VALUES (?,?,?,?)",
                     (client_options[client.value], datetime.now().strftime('%Y-%m-%d'), amount.value, category.value))
            conn.commit()
            print(f"Sale added! Client: {client.value}, amount: {amount.value} RUB")

    button.on_click(on_button_click)
    display(client, amount, category, button, output)

# Меню
print("=" * 50)
print("CRM SYSTEM FOR RETAIL")
print("=" * 50)

menu = widgets.Dropdown(
    options=['Show KPI', 'Client List', 'Add Client', 'Add Sale'],
    description='Menu:'
)
display(menu)

def on_menu_change(change):
    clear_output(wait=True)
    display(menu)
    if change['new'] == 'Show KPI':
        show_kpi()
    elif change['new'] == 'Client List':
        show_clients()
    elif change['new'] == 'Add Client':
        add_client_ui()
    elif change['new'] == 'Add Sale':
        add_sale_ui()

menu.observe(on_menu_change, names='value')

Dropdown(description='Menu:', options=('Show KPI', 'Client List', 'Add Client', 'Add Sale'), value='Show KPI')

LTV: 14,667 RUB
Average check: 8,800 RUB
Conversion: 100.0%
Total revenue: 88,000 RUB
